In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv
/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv


# import 

In [2]:
import os
import gc
import math
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup
)


# 2) Reproducibility


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

# 3) Load datasets


In [4]:
SENTENCE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv"
ARTICLE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv"


In [5]:
sent_df = pd.read_csv(SENTENCE_PATH)
art_df = pd.read_csv(ARTICLE_PATH)

print("Sentence-level shape:", sent_df.shape)
print("Article-level shape:", art_df.shape)

print(sent_df.head())
print(art_df.head())


Sentence-level shape: (7344, 3)
Article-level shape: (1018, 3)
   Unnamed: 0                                           sentence  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  In terms of linguistics, a program must be abl...      0
2           2  Of course each language has its own forms of a...      0
3           3  Programs can use several strategies for dealin...      0
4           4  As formidable as the task of extracting the co...      0
   Unnamed: 0                                            article  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  There are a variety of emerging applications f...      0
2           2  As each new means of communication and social ...      0
3           3  These suggestions include:, Learn about the pu...      0
4           4  In recent years there has been growing concern...      0


# 4) Robust column detection


In [6]:
# This is a helper function which can be handy for future

def detect_text_label_columns(df):
    cols = [c.lower() for c in df.columns]
    original = list(df.columns)

    text_candidates = ["text", "sentence", "article", "content", "response"]
    label_candidates = ["label", "class", "target", "generated"]

    text_col = None
    label_col = None

    for cand in text_candidates:
        for i, c in enumerate(cols):
            if cand == c or cand in c:
                text_col = original[i]
                break
        if text_col is not None:
            break

    for cand in label_candidates:
        for i, c in enumerate(cols):
            if cand == c or cand in c:
                label_col = original[i]
                break
        if label_col is not None:
            break

    if text_col is None:
        for c in original:
            if df[c].dtype == object:
                text_col = c
                break

    if label_col is None:
        for c in original:
            if pd.api.types.is_numeric_dtype(df[c]):
                uniq = sorted(df[c].dropna().unique().tolist())
                if set(uniq).issubset({0, 1}):
                    label_col = c
                    break

    return text_col, label_col


# 5)Basic cleaning

In [7]:
sent_text_col, sent_label_col = detect_text_label_columns(sent_df)
art_text_col, art_label_col = detect_text_label_columns(art_df)

print("Sentence-level:", sent_text_col, sent_label_col)
print("Article-level:", art_text_col, art_label_col)


Sentence-level: sentence class
Article-level: article class


In [8]:
def prepare_dataframe(df, text_col, label_col):
    df = df[[text_col, label_col]].copy()
    df.columns = ["text", "label"]
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(int)
    df = df.dropna()
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df

sent_df = prepare_dataframe(sent_df, sent_text_col, sent_label_col)
art_df = prepare_dataframe(art_df, art_text_col, art_label_col)

print(sent_df["label"].value_counts())
print(art_df["label"].value_counts())


label
1    4008
0    3336
Name: count, dtype: int64
label
0    509
1    509
Name: count, dtype: int64


# 6) Train/validation/test split


In [9]:
def stratified_split(df, test_size=0.15, val_size=0.15, seed=42):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=seed
    )
    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size / (1 - test_size),
        stratify=train_df["label"],
        random_state=seed
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

sent_train, sent_val, sent_test = stratified_split(sent_df)
art_train, art_val, art_test = stratified_split(art_df)

print(len(sent_train), len(sent_val), len(sent_test))
print(len(art_train), len(art_val), len(art_test))


5140 1102 1102
712 153 153


# 7) Metrics and evaluation


In [10]:
def compute_metrics(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm
    }

    if y_prob is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        except:
            metrics["roc_auc"] = None
    else:
        metrics["roc_auc"] = None

    return metrics


def print_metrics(title, metrics, y_true=None, y_pred=None):
    print(f"\n===== {title} =====")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")
    if metrics["roc_auc"] is not None:
        print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")
    print("Confusion Matrix:")
    print(metrics["confusion_matrix"])
    if y_true is not None and y_pred is not None:
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, digits=4))


# 8) Dataset classes


In [11]:
class TransformerTextDataset(Dataset):
    # may be need to change this line of code.
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
